# Fading

> The channel coefficient h of a transmission, path loss included: the fading models.

In [ ]:
#| default_exp channel.fading

In [ ]:
#| hide
from nbdev.showdoc import *

A `Fading` gives the channel coefficient `h` of a transmission: the first half of a channel. `h` is the whole
coefficient, in the broad sense of a *fading channel*: the path loss and shadowing (`|h|²` is the power gain), the
small-scale fading, and the phase. The second half applies `h` to the transmitted symbols, `y = h √P x + n`, at
the link level (see `channel.awgn`). Channels at both levels use it: `SystemLevelChannel` (see `radio.system.channel`) and `LinkLevelChannel`
(see `radio.link.channel`).

* `NoFading`: a constant `h`.
* `RayleighFading`: `h ~ CN(0, gain)`, drawn anew for every transmission.
* `GilbertElliottFading`: another fading, weakened during bad episodes in time.
* Ray-traced channels, `RayTracedFading` and `RayTracedPathsFading`: see `channel.ray_tracing`.

A `Fading` is a block (see `block`): called on a step's transmissions and links, it gives their `h`, one row per
transmission and one column per subcarrier. A new fading only needs `coefficient`, or `response` for subcarriers.

In [ ]:
#| export
from __future__ import annotations

import bisect
import math
from abc import ABC, abstractmethod
from typing import Sequence

import numpy as np

from comm_core.block import Block
from comm_core.core import Transmission
from comm_core.net.topology import Link

In [ ]:
#| export
class Fading(Block, ABC):
    "The channel coefficient h of a transmission. As a block: the h of a step's transmissions, at each subcarrier."

    outputs = ('h',)

    @abstractmethod
    def coefficient(
        self,
        transmission: Transmission, # The transmission (its sender and receiver)
        link: Link, # Its link
    ) -> complex:
        "h of this transmission (math.inf: the nodes are at the same place, which means a perfect link)."

    def response(
        self,
        transmission: Transmission, # The transmission (its sender and receiver)
        link: Link, # Its link
    ) -> np.ndarray:
        "h at each subcarrier (a frequency-selective fading); by default flat: one value, `coefficient`."
        return np.array([complex(self.coefficient(transmission, link))])

    def call(
        self,
        transmission: Sequence[Transmission], # The transmissions
        link: Sequence[Link], # Their links
    ) -> np.ndarray:
        "h of each transmission at each subcarrier, (n, K) complex: one `response` per row (a row of inf: the nodes at the same place)."
        rows = [np.asarray(self.response(t, l), np.complex128).reshape(-1) for t, l in zip(transmission, link)]
        if not rows:
            return np.zeros((0, 1), np.complex128)
        K = max(len(r) for r in rows)
        rows = [np.full(K, r[0]) if len(r) == 1 and np.isinf(r).any() else r for r in rows]
        if any(len(r) != K for r in rows):
            raise ValueError(f"responses with different numbers of subcarriers: {sorted({len(r) for r in rows})}")
        return np.stack(rows)


def effective_coefficient(
    h, # h of a transmission at its subcarriers (one value: flat)
) -> complex:
    "One coefficient for a transmission: h if flat, else √(mean |h|²) (the same power); math.inf if the nodes are at the same place."
    h = np.asarray(h, np.complex128).reshape(-1)
    if np.isinf(h).any():
        return complex(math.inf)
    return complex(h[0]) if h.size == 1 else complex(np.sqrt(np.mean(np.abs(h) ** 2)))

class NoFading(Fading):
    "The same coefficient for every transmission."

    def __init__(
        self,
        h: complex = 1.0, # The coefficient
    ):
        self.h = h

    def coefficient(self, transmission, link):
        return self.h


class RayleighFading(Fading):
    "h ~ CN(0, gain), drawn anew for every transmission (constant during a packet)."

    def __init__(
        self,
        gain: float = 1.0, # Average power gain E|h|^2 (the path loss)
        seed: int | None = None, # Seed of the draws
    ):
        self.gain = gain
        self.rng = np.random.default_rng(seed)

    def reset(self, seed=None):
        if seed is not None:
            self.rng = np.random.default_rng(seed)

    def coefficient(self, transmission, link):
        return complex(self.rng.standard_normal(), self.rng.standard_normal()) * math.sqrt(self.gain / 2)

In [ ]:
show_doc(Fading.coefficient)

In [ ]:
show_doc(Fading.call)

In [ ]:
t = Transmission(message=None, sender='a', receiver='b', start_time=0.0)
f = RayleighFading(gain=2.0, seed=0)
hs = np.array([f.coefficient(t, None) for _ in range(20_000)])
assert abs((np.abs(hs) ** 2).mean() - 2.0) < 0.05
f.reset(seed=0); assert f.coefficient(t, None) == hs[0]
assert NoFading(0.5j).coefficient(t, None) == 0.5j

In [ ]:
# as a block: the h of a batch, row by row, the same draws as one at a time
pairs = [(Transmission(message=None, sender='a', receiver=r, start_time=0.0), Link('a', r)) for r in 'bcd']
f.reset(seed=1); one_by_one = [f.coefficient(t, l) for t, l in pairs]
f.reset(seed=1); h = f(*zip(*pairs))
assert h.shape == (3, 1) and h[:, 0].tolist() == one_by_one
assert f.outputs == ('h',) and f.inputs == ('transmission', 'link')
assert effective_coefficient([3.0, 4j]) == np.sqrt(12.5) and effective_coefficient([1j]) == 1j
assert effective_coefficient(NoFading(math.inf)(pairs[0][:1], pairs[0][1:])[0]) == math.inf

### Time-correlated impairments

A fading can depend on time: a transmission carries its time (`transmission.start_time`, the network's time, or
the time of each attempt in a `FragmentingChannel`).

`GilbertElliottFading` uses it to weaken another fading by `penalty_db` during bad episodes. These are the episodes
of a two-state Markov process (good ↔ bad, with exponential durations), e.g. bursts of interference or a body
blocking the line of sight. The episodes are fixed by the seed, so two methods compared with the same seed see the
same bursts (common random numbers).

In [ ]:
#| export
class GilbertElliottFading(Fading):
    "Another fading, weakened by `penalty_db` during the bad episodes of a two-state Markov process in time."

    def __init__(
        self,
        fading: Fading, # The fading in good times
        mean_good_s: float = 3.0, # Mean time between bad episodes
        mean_bad_s: float = 0.6, # Mean length of a bad episode
        penalty_db: float = 30.0, # How much weaker the channel is during one (SINR drop)
        seed: int | None = None, # Seed of the episodes
    ):
        self.fading, self.mean_good_s, self.mean_bad_s, self.penalty_db = fading, mean_good_s, mean_bad_s, penalty_db
        self._start(seed)

    def _start(self, seed):
        self.rng, self.edges, self._end, self._bad = np.random.default_rng(seed), [], 0.0, False

    def _extend(self, until: float) -> None:
        while self._end <= until:                           # draw the episodes as far as needed (good at time 0)
            self._end += self.rng.exponential(self.mean_bad_s if self._bad else self.mean_good_s)
            self.edges.append(self._end)
            self._bad = not self._bad

    def bad(
        self,
        time: float, # A time
    ) -> bool:
        "Whether `time` is in a bad episode."
        self._extend(time)
        return bisect.bisect_right(self.edges, time) % 2 == 1

    def episodes(
        self,
        until: float, # Up to this time
    ) -> list:
        "The bad episodes up to `until`: [(start, end), ...]."
        self._extend(until)
        e = self.edges
        return [(e[i], min(e[i + 1], until)) for i in range(0, len(e) - 1, 2) if e[i] < until]

    def reset(self, seed=None):
        "`seed` draws new episodes (and reseeds the inner fading)."
        self.fading.reset(seed)
        if seed is not None:
            self._start(seed)

    def set_positions(self, positions) -> None:
        if hasattr(self.fading, 'set_positions'):
            self.fading.set_positions(positions)

    def _scale(self, transmission) -> float:
        return 10 ** (-self.penalty_db / 20) if self.bad(transmission.start_time) else 1.0

    def coefficient(self, transmission, link):
        h = self.fading.coefficient(transmission, link)
        return h if math.isinf(abs(h)) else h * self._scale(transmission)

    def response(self, transmission, link):
        h = self.fading.response(transmission, link)
        return h if np.isinf(h).any() else h * self._scale(transmission)

In [ ]:
show_doc(GilbertElliottFading.bad)

In [ ]:
show_doc(GilbertElliottFading.episodes)

In [ ]:
ge = GilbertElliottFading(NoFading(1.0), mean_good_s=3.0, mean_bad_s=0.6, penalty_db=30, seed=0)
times = np.arange(0, 2000, 0.05)
bad = np.array([ge.bad(x) for x in times])
assert not ge.bad(0.0) and abs(bad.mean() - 0.6 / 3.6) < 0.03             # bad 1/6 of the time
at = lambda x: Transmission(message=None, sender='a', receiver='b', start_time=x)
x_bad, x_good = times[bad][0], times[~bad][0]
assert abs(ge.coefficient(at(x_bad), None) - 10 ** -1.5) < 1e-12 and ge.coefficient(at(x_good), None) == 1.0
assert all(ge.bad((s + e) / 2) for s, e in ge.episodes(100))
same = GilbertElliottFading(NoFading(), seed=0)
assert same.episodes(500) == ge.episodes(500)                               # the same seed: the same bursts
ge.reset(seed=1)
assert ge.episodes(500) != same.episodes(500)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()